<a href="https://colab.research.google.com/github/naokityokoyama/fake_news_hdc/blob/main/ML_TFIDF_W2V_BOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install unidecode evaluate num2words -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 17.2 MB/s eta 0:00:00


In [ ]:
!pip install gensim

In [ ]:
import zipfile
import os
from unidecode import unidecode
import string
from num2words import num2words
import re
import numpy as np
import pandas as pd
from typing import Union, Literal
from tqdm.notebook import tqdm
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import time
import re
import gensim.downloader as api
import joblib
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#build dataset
def build_dataset(isot, covid, fever):
  if isot:
    df = pd.read_csv('/content/drive/MyDrive/uff/isot.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)
  elif covid:
    df = pd.read_csv('/content/drive/MyDrive/uff/covid.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)
  elif fever:
    df = pd.read_csv('/content/drive/MyDrive/uff/fever.csv')
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df = build_dataset(isot=False, covid=True, fever=False)
X_T, X_t, y_T, y_t = train_test_split(df['frase'], df['target'], test_size=0.30, random_state = 42)

In [ ]:
'treino->', len(X_T), 'teste ->' , len(y_t), 'size ->', df.shape[0]

('treino->', 4182, 'teste ->', 1793, 'size ->', 5975)

In [ ]:
def batch(dataset, batch=True, size=5000):
  # Definir o tamanho da amostra
  if batch:
    sample_size = size

    # Criar uma amostra balanceada
    dataset = dataset.groupby("target", group_keys=False).apply(lambda x: resample(x, n_samples=sample_size // dataset["target"].nunique(), random_state=42))
    dataset = dataset.reset_index(drop=True)
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    return dataset

  else:
    return dataset

In [ ]:
# df = batch(df, batch=True, size=1000)

In [ ]:
df.shape

(5975, 3)

In [ ]:
def n2w(texto:str)->str:
  padrao = r"\d+"
  numeros = re.findall(padrao, texto)
  for numero in numeros:
    # Check if the number is within the num2words limit
    if abs(int(numero)) < 10**27: # The limit is 10^27 for num2words
        try:
            extenso = num2words(numero, lang='pt')
            texto = texto.replace(numero, extenso)
        except ValueError: # Handle potential errors during conversion
            pass # Keep the original number if conversion fails
    else:
        texto = texto.replace(numero, "[LARGE_NUMBER]") # Replace very large numbers with a placeholder
  return texto

In [ ]:
for repet in tqdm(range(2)):  #bug para rodar 2x
  df['frase'] = df['frase'].str.lower()
  df['frase'] = df['frase'].str.replace(f"[{string.punctuation}]", "", regex=True)
  df['frase'] = df['frase'].apply(lambda x: ' '.join(x.split()))
  df['frase'] = df['frase'].str.replace('"', '').str.replace('\\', '')
  #df['frase'] = df['frase'].apply(n2w)
  df['frase'] = df['frase'].apply(unidecode)

  0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
tfidf = TfidfVectorizer(analyzer='word')
bow = CountVectorizer()
model_RL = LogisticRegression()
model_RF = RandomForestClassifier()
model_SVC = SVC()
model_TREE = DecisionTreeClassifier()
model_KNN = KNeighborsClassifier()

In [ ]:
lst_models = [model_RL,model_RF, model_SVC, model_TREE, model_KNN]

In [ ]:
batch_size = 1000
num_samples = df.shape[0]

In [ ]:
def train(model:str, encoder:str):

  lst_acc = []
  lst_f1 = []
  lst_recall = []
  lst_precision = []
  lst_start = []
  lst_end = []
  lst_cm = []
  lst_roc = []



  for i in tqdm(range(0, num_samples, batch_size)):
    #start time
    start = time.perf_counter()
    X = df['frase'][i:i+batch_size]
    y = df['target'][i:i+batch_size]

    if encoder=='tfidf':
      X_tfidf = tfidf.fit_transform(X)
      X_ = X_tfidf.toarray()
    if encoder=='bow':
      X_bow = bow.fit_transform(X)
      X_ = X_bow.toarray()

    X_train, X_test, y_train, y_test = train_test_split(X_, y, test_size=0.30, random_state = 42)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # # METRICS
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred)

    #end
    end = time.perf_counter()

    # #LIST ADD
    lst_acc.append(acc)
    lst_f1.append(f1)
    lst_precision.append(precision)
    lst_recall.append(recall)
    lst_cm.append(cm)
    lst_roc.append(roc)
    lst_start.append(start)
    lst_end.append(end)

    filesave = 'model.joblib'
    joblib.dump(model, filesave)
    tamanho_bytes = os.path.getsize(filesave)
    tamanho_mb = tamanho_bytes / (1024 * 1024)



  print ('-- REPORT --')
  print (f'-- {model} --')
  print ('ACCURACY ->', np.mean(lst_acc))
  print ('F1 ->', np.mean(lst_f1))
  print ('Precision ->', np.mean(lst_precision))
  print ('Recall ->', np.mean(lst_recall))
  print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
  print (f'O Tamanho do {model} {tamanho_mb:.3f}')
  print(f"Confusion Matrix of ", np.mean(lst_cm, axis=0))
  print(f"ROC of ", np.nanmean(lst_roc))
  print(f"Size train", len(X_T))
  print(f"Size test", len(X_t))
  print ('\n')
  print ('------------------------------------------------')

In [ ]:
for i in tqdm(lst_models):
  train(i, encoder='bow')

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

-- REPORT --
-- LogisticRegression() --
ACCURACY -> 0.8600777398558969
F1 -> 0.6362072156727834
Precision -> 0.830253169314496
Recall -> 0.5177354851267895
Time total: 2.475292 s
O Tamanho do LogisticRegression() 0.023
Confusion Matrix of  [[220.33333333   7.5       ]
 [ 34.33333333  36.66666667]]
ROC of  0.7423722948608531
Size train 4182
Size test 1793


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

-- REPORT --
-- RandomForestClassifier() --
ACCURACY -> 0.8092434584755402
F1 -> 0.3623735760858529
Precision -> 0.869211373845241
Recall -> 0.23402493239449762
Time total: 3.695101 s
O Tamanho do RandomForestClassifier() 2.509
Confusion Matrix of  [[225.16666667   2.66666667]
 [ 54.33333333  16.66666667]]
ROC of  0.6112168366029923
Size train 4182
Size test 1793


------------------------------------------------


...

-- REPORT --
-- SVC() --
ACCURACY -> 0.8544292756920743
F1 -> 0.6059160316753714
Precision -> 0.8479853435325194
Recall -> 0.4728565516608994
Time total: 4.489177 s
O Tamanho do SVC() 9.732
Confusion Matrix of  [[221.83333333   6.        ]
 [ 37.5         33.5       ]]
ROC of  0.7232513813892533
Size train 4182
Size test 1793


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

-- REPORT --
-- DecisionTreeClassifier() --
ACCURACY -> 0.8182783466059916
F1 -> 0.5922676198600775
Precision -> 0.6430300161037527
Recall -> 0.5625193043671305
Time total: 0.938480 s
O Tamanho do DecisionTreeClassifier() 0.016
Confusion Matrix of  [[204.33333333  23.5       ]
 [ 30.83333333  40.16666667]]
ROC of  0.729662039601971
Size train 4182
Size test 1793


------------------------------------------------


  0%|          | 0/6 [00:00<?, ?it/s]

-- REPORT --
-- KNeighborsClassifier() --
ACCURACY -> 0.7730925293894576
F1 -> 0.11673440621244136
Precision -> 0.7640056022408963
Recall -> 0.06835374770157379
Time total: 0.493692 s
O Tamanho do KNeighborsClassifier() 15.007
Confusion Matrix of  [[226.33333333   1.5       ]
 [ 66.33333333   4.66666667]]
ROC of  0.5309359902323288
Size train 4182
Size test 1793


------------------------------------------------


W2V

In [ ]:
import re
import gensim.downloader as api

In [ ]:
#download corpus w2v 300d
wv = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
#encoding
def tokenize(text: str):
    return re.findall(r"\b\w+\b", text.lower())

def sentence_embedding(text: str) -> np.ndarray:
    tokens = tokenize(text)
    vecs = [wv[t] for t in tokens if t in wv]
    if not vecs:
        return np.zeros(wv.vector_size, dtype=np.float32)
    #calculando a media (Mean Pooling)
    return np.mean(vecs, axis=0).astype(np.float32)

def texts_to_matrix(texts):
    return np.vstack([sentence_embedding(t) for t in tqdm(texts)])

In [ ]:
embedding = texts_to_matrix(df['text'])

  0%|          | 0/5975 [00:00<?, ?it/s]

In [ ]:
y = df['target'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(embedding, y, test_size=0.30, random_state = 42)

In [ ]:
model_RL = LogisticRegression()
model_RF = RandomForestClassifier()
model_SVC = SVC()
model_TREE = DecisionTreeClassifier()
model_KNN = KNeighborsClassifier()

In [ ]:
lst_models = [model_RL, model_RF, model_SVC, model_TREE, model_KNN]

In [ ]:
lst_acc = []
lst_f1 = []
lst_recall = []
lst_precision = []
lst_cm = []
lst_roc = []
lst_start = []
lst_end = []
lst_size = []

for model in tqdm(lst_models):
  #start time
  start = time.perf_counter()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  # # METRICS
  acc = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  recall = recall_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred)
  cm = confusion_matrix(y_test, y_pred)
  roc = roc_auc_score(y_test, y_pred)

  #end
  end = time.perf_counter()

  filesave = 'model.joblib'
  joblib.dump(model, filesave)
  tamanho_bytes = os.path.getsize(filesave)
  tamanho_mb = tamanho_bytes / (1024 * 1024)
  lst_size.append(tamanho_mb)

  size_full =np.sum(lst_size)
  # #LIST ADD
  lst_acc.append(acc)
  lst_f1.append(f1)
  lst_precision.append(precision)
  lst_recall.append(recall)
  lst_cm.append(cm)
  lst_roc.append(roc)
  lst_start.append(start)
  lst_end.append(end)



  print ('-- REPORT --')
  print (f'-- {model} --')
  print ('ACCURACY ->', np.mean(lst_acc))
  print ('F1 ->', np.mean(lst_f1))
  print ('Precision ->', np.mean(lst_precision))
  print ('Recall ->', np.mean(lst_recall))
  print(f"Confusion Matrix of ", np.mean(lst_cm, axis=0))
  print(f"ROC of ", np.nanmean(lst_roc))
  print(f"Time total: {lst_end[-1] - lst_start[0]:.6f} s")
  print (f'O Tamanho do {model} {size_full:.3f}')
  print ('\n')
  print ('------------------------------------------------')

  0%|          | 0/5 [00:00<?, ?it/s]

-- REPORT --
-- LogisticRegression() --
ACCURACY -> 0.8616843279419967
F1 -> 0.6666666666666666
Precision -> 0.8378378378378378
Recall -> 0.5535714285714286
Confusion Matrix of  [[1297.   48.]
 [ 200.  248.]]
ROC of  0.7589418481147105
Time total: 0.329055 s
O Tamanho do LogisticRegression() 0.003


------------------------------------------------
-- REPORT --
-- RandomForestClassifier() --
ACCURACY -> 0.852760736196319
F1 -> 0.6138975966562173
Precision -> 0.8899715504978662
Recall -> 0.4765625
Confusion Matrix of  [[1315.5   29.5]
 [ 234.5  213.5]]
ROC of  0.7273147072490707
Time total: 8.048782 s
O Tamanho do RandomForestClassifier() 5.078


------------------------------------------------
-- REPORT --
-- SVC() --
ACCURACY -> 0.870422011526306
F1 -> 0.6729651056406096
Precision -> 0.8887898979311721
Recall -> 0.5558035714285715
Confusion Matrix of  [[1311.66666667   33.33333333]
 [ 199.          249.        ]]
ROC of  0.765510211984422
Time total: 10.006704 s
O Tamanho do SVC() 8.83

bert e roberta

In [ ]:
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
df_ = df[['frase', 'target']]

In [ ]:
df_ = df_.rename(columns={'target': 'label'})

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoConfig
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np

# Garantindo que df_ esteja definido
df_ = df[['frase', 'target']].rename(columns={'target': 'label'})

train_df, val_df = train_test_split(df_, test_size=0.3, random_state=42)

hf_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df, preserve_index=False)
})

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return clf_metrics.compute(predictions=predictions, references=labels)

def train_and_evaluate_model(model_name, dataset, output_dir):
    print(f"\n{'='*50}")
    print(f"Iniciando pipeline para o modelo: {model_name}")
    print(f"{'='*50}\n")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, force_download=True)

    def tokenize_function(examples):
        return tokenizer(examples["frase"], truncation=True, padding=False, max_length=128)

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    config = AutoConfig.from_pretrained(model_name, num_labels=2)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        ignore_mismatched_sizes=True,
        force_download=True
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        push_to_hub=False,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print(f"Treinando {model_name}...")
    trainer.train()

    print(f"\nAvaliação final do {model_name}:")
    eval_results = trainer.evaluate()

    predictions_output = trainer.predict(tokenized_datasets["validation"])

    y_pred = np.argmax(predictions_output.predictions, axis=1)
    y_prob = torch.nn.functional.softmax(torch.tensor(predictions_output.predictions), dim=-1).numpy()[:, 1]
    y_true = predictions_output.label_ids

    cm = confusion_matrix(y_true, y_pred)
    roc = roc_auc_score(y_true, y_prob)
    eval_results['confusion_matrix'] = cm
    eval_results['roc_auc'] = roc

    return eval_results


In [ ]:
modelos_para_testar = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base"
}

resultados_finais = {}

for nome_amigavel, identificador_hf in modelos_para_testar.items():
    diretorio_saida = f"./resultados_{nome_amigavel.lower()}"

    metricas = train_and_evaluate_model(
        model_name=identificador_hf,
        dataset=hf_dataset,
        output_dir=diretorio_saida
    )

    resultados_finais[nome_amigavel] = metricas

# Exibir Relatório Final
print("\n" + "="*50)
print("COMPARAÇÃO DE RESULTADOS")
print("="*50)
for modelo, metricas in resultados_finais.items():
    print(f"Modelo: {modelo}")
    print(f"  - Acurácia: {metricas.get('eval_accuracy', 0):.4f}")
    print(f"  - Precisão: {metricas.get('eval_precision', 0):.4f}")
    print(f"  - Recall:   {metricas.get('eval_recall', 0):.4f}")
    print(f"  - F1-Score: {metricas.get('eval_f1', 0):.4f}")
    if 'roc_auc' in metricas:
        print(f"  - ROC AUC:  {metricas.get('roc_auc', 0):.4f}")
    print("\n")

    # Imprimir a matriz de confusão
    print("  - Matriz de Confusão:")
    cm = metricas.get('confusion_matrix')
    if cm is not None:
        # Formatação simples e visual
        print(f"      [ {cm[0][0]:4d} | {cm[0][1]:4d} ] (Reais: 0)")
        print(f"      [ {cm[1][0]:4d} | {cm[1][1]:4d} ] (Reais: 1)")
        print("        (Pred:0) (Pred:1)")
    else:
        print("      Não disponível")
    print("\n")



Iniciando pipeline para o modelo: bert-base-uncased



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.283620,0.909649,0.815490,0.832558,0.799107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Avaliação final do roberta-base:


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.283620,1,0.909649,0.815490,0.832558,0.799107



COMPARAÇÃO DE RESULTADOS
Modelo: BERT
  - Acurácia: 0.9147
  - Precisão: 0.8172
  - Recall:   0.8482
  - F1-Score: 0.8324
  - ROC AUC:  0.9630


  - Matriz de Confusão:
      [ 1260 |   85 ] (Reais: 0)
      [   68 |  380 ] (Reais: 1)
        (Pred:0) (Pred:1)


Modelo: RoBERTa
  - Acurácia: 0.9096
  - Precisão: 0.8326
  - Recall:   0.7991
  - F1-Score: 0.8155
  - ROC AUC:  0.9528


  - Matriz de Confusão:
      [ 1273 |   72 ] (Reais: 0)
      [   90 |  358 ] (Reais: 1)
        (Pred:0) (Pred:1)




In [ ]:
import pandas as pd

# Exibindo detalhadamente a configuração da Matriz de Confusão e ROC AUC para BERT e RoBERTa
for modelo, metricas in resultados_finais.items():
    print(f"--- Configuração de Matriz para {modelo} ---")
    if 'confusion_matrix' in metricas:
        cm = metricas['confusion_matrix']
        display(pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1']))
    else:
        print(f"A matriz de confusão não foi encontrada nos resultados de {modelo}.")

    if 'roc_auc' in metricas:
        print(f"ROC AUC: {metricas['roc_auc']:.4f}")
    print('\n')


--- Configuração de Matriz para BERT ---


,Pred 0,Pred 1
Real 0,22921,1012
Real 1,3603,5407


ROC AUC: 0.9003


--- Configuração de Matriz para RoBERTa ---


,Pred 0,Pred 1
Real 0,22699,1234
Real 1,3572,5438


ROC AUC: 0.8889


